# 中证800 V51 Factor Increment Mining 实验

目标：在 V46 强 baseline 上做单因子增量挖掘，不换 label、不换模型框架、不改组合逻辑。

实验闭环：

1. 先从聚宽 API 重建 V4/V46 中证800月频基础数据，不依赖旧 CSV。
2. 再从聚宽因子库按小批量拉取候选增量因子。
3. 对每个候选因子做 OOS 单因子审计：coverage、top10/top30 月度 alpha、年度稳定性、最差月份。
4. 在同一训练环境下导出 `V46 baseline` 与 `V46 + each_factor` 的 direct pkl。
5. 通过 JoinQuant 回测文件加载 pkl 验证真实组合效果。

默认启用现金流质量、盈利质量、经营效率三组未验证候选因子；`turnover_volatility` 只保留为手动对照，不再默认重测。若某个因子加入 V46 后回测改善，再进入 candidate pool；否则不保留。

In [ ]:
import gc
import os
import pickle
import warnings
import datetime

try:
    from jqdata import *
    from jqfactor import get_factor_values
except Exception:
    # Local syntax checks do not have JoinQuant APIs. Rebuild/fetch paths are only used in JoinQuant research.
    pass

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)

DATA_PATH = "train_csi800_factor_v40_data_enhancement.csv"

# V51 should be self-contained by default: rebuild the base V4/V46 monthly dataset first, then fetch new factors.
REBUILD_DATA = True
USE_REBUILT_DATA_FOR_TRAINING = True
REBUILD_DATA_START = "2019-01-01"
REBUILD_DATA_END_FOR_LABEL = "2026-05-31"
REBUILD_DATA_TAG = "{}_{}".format(REBUILD_DATA_START.replace("-", ""), REBUILD_DATA_END_FOR_LABEL.replace("-", ""))
REBUILD_DATA_OUTPUT_PATH = "train_csi800_factor_v40_data_enhancement_{}.csv".format(REBUILD_DATA_TAG)
REBUILD_MANIFEST_PATH = "train_csi800_factor_v40_data_enhancement_{}_manifest.csv".format(REBUILD_DATA_TAG)
REBUILD_FORCE_OVERWRITE = True
OUT_DIR = "csi800_ml_v51_factor_increment_mining_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_COL = "alpha_1m"
TRAIN_START = "2019-01-01"
TRAIN_END = "2025-03-31"
LABEL_END = "2025-03-31"
REQUIRE_LABEL_END_WITHIN_TRAIN = False

BENCHMARK = "000906.XSHG"
TOP_N_CANDIDATES = 30
STOCK_NUM = 10
INDUSTRY_CAP_RATIO = 0.20
CORR_THRESHOLD = 0.70
INNER_VALID_FRAC = 0.20
INNER_VALID_MIN_MONTHS = 6
SEED = 42
FIXED_NUM_BOOST_ROUND = 120
MIN_HISTORY_MONTHS = 24
TOP_LIST = [10, 30]

FETCH_FACTORS_IF_CACHE_MISSING = True
FORCE_REFETCH_FACTORS = False
FACTOR_CHUNK_SIZE = 1
STOCK_CHUNK_SIZE = 150
GC_EVERY_N_FETCH_DATES = 2
GC_EVERY_N_EVAL_MONTHS = 6

EXPORT_BASELINE_MODEL = True
EXPORT_INCREMENT_MODELS = True
EXPORT_UNSELECTED_MODELS = False

print("DATA_PATH =", DATA_PATH)
print("REBUILD_DATA =", REBUILD_DATA, "REBUILD_DATA_OUTPUT_PATH =", REBUILD_DATA_OUTPUT_PATH)
print("OUT_DIR =", OUT_DIR)


In [ ]:
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m",
    "ts_Rank1M_rank_chg_1m",
]

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

BASE_CANDIDATE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS


In [ ]:
# =========================
# Required base data rebuild block
# =========================
# V51 default path: REBUILD_DATA=True. It rebuilds base CSI800 V4/V46 data before incremental factor mining.

UNIVERSE_NAME = "CSI800"
UNIVERSE_INDEX = "000906.XSHG"
MIN_LISTING_DAYS = 180
V4_DATA_START = REBUILD_DATA_START
V4_DATA_END_FOR_LABEL = REBUILD_DATA_END_FOR_LABEL
V4_DATA_FILE = REBUILD_DATA_OUTPUT_PATH

V4_PRICE_PATH_COLS = [
    "px_ret_5", "px_ret_20", "px_ret_60", "px_ret_120",
    "px_close_to_ma20", "px_close_to_ma60", "px_ma20_to_ma60",
    "px_volatility_20", "px_volatility_60", "px_drawdown_20", "px_drawdown_60", "px_drawdown_120",
    "px_up_day_ratio_20", "px_new_high_distance_60", "px_new_low_distance_60",
    "px_skew_20", "px_kurt_20",
]

TRADE_LIQUIDITY_COLS = [
    "liq_money_mean_20", "liq_money_mean_60", "liq_money_ratio_20_60",
    "liq_volume_mean_20", "liq_volume_ratio_20_60",
    "liq_amplitude_mean_20", "liq_amplitude_mean_60",
    "liq_paused_count_20", "liq_paused_count_60",
    "liq_low_money_days_20", "liq_limit_up_count_20", "liq_limit_down_count_20", "liq_one_price_limit_count_20",
]

CONTEXT_COLS = [
    "ctx_industry_ret_20", "ctx_industry_ret_60",
    "ctx_stock_minus_industry_ret_20", "ctx_stock_minus_industry_ret_60",
    "ctx_stock_rank_industry_ret_20", "ctx_stock_rank_industry_volatility_20",
    "ctx_market_ret_20", "ctx_market_ret_60", "ctx_market_volatility_20",
]

CORE_TEMPORAL_FACTORS = [
    "book_to_price_ratio", "earnings_yield", "cash_flow_to_price_ratio",
    "Rank1M", "sharpe_ratio_60", "VOSC", "MFI14",
]

def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def require_joinquant_api():
    # Directly probe the JoinQuant API instead of inspecting notebook namespaces.
    try:
        get_trade_days(end_date="2019-01-02", count=1)
    except NameError:
        raise RuntimeError("data rebuild requires JoinQuant research runtime: get_trade_days is not available")
    except Exception:
        # API exists; date/account related errors should surface in the actual caller.
        pass


def get_period_date(period, start_date, end_date):
    require_joinquant_api()
    trade_days = pd.to_datetime(get_trade_days(start_date=start_date, end_date=end_date))
    if len(trade_days) == 0:
        return []
    if period != "M":
        raise ValueError("V4 data pipeline only supports monthly period M")

    dates = []
    last_key = None
    for d in trade_days:
        key = d.strftime("%Y-%m")
        if key != last_key:
            dates.append(d.strftime("%Y-%m-%d"))
            last_key = key
    return dates


def get_previous_trade_date(date):
    require_joinquant_api()
    trade_days = pd.to_datetime(get_trade_days(end_date=date, count=2))
    if len(trade_days) < 2:
        return None
    return trade_days[-2].strftime("%Y-%m-%d")


def delect_stop(stocks, begin_date, n=180):
    stock_list = []
    begin_dt = pd.Timestamp(begin_date).to_pydatetime()
    for stock in stocks:
        info = get_security_info(stock)
        if info is None:
            continue
        if info.start_date <= (begin_dt - datetime.timedelta(days=n)).date():
            stock_list.append(stock)
    return stock_list


def filter_paused_stock_by_date(stock_list, date):
    if len(stock_list) == 0:
        return []
    try:
        paused_df = get_price(
            stock_list,
            end_date=date,
            frequency="daily",
            fields=["paused"],
            count=1,
            skip_paused=False,
            panel=False,
            fill_paused=True,
        )
    except Exception:
        return stock_list

    if paused_df is None or paused_df.empty or "paused" not in paused_df.columns:
        return stock_list

    paused_map = paused_df.groupby("code")["paused"].last()
    return [
        stock for stock in stock_list
        if (stock not in paused_map.index) or (not bool(paused_map.loc[stock]))
    ]


def get_stock(stock_pool, feature_date):
    require_joinquant_api()
    if stock_pool == "CSI800":
        stock_list = get_index_stocks(UNIVERSE_INDEX, feature_date)
    elif stock_pool == "HS300":
        stock_list = get_index_stocks("000300.XSHG", feature_date)
    elif stock_pool == "ZZ1000":
        stock_list = get_index_stocks("000852.XSHG", feature_date)
    elif stock_pool == "A":
        stock_list = get_index_stocks("000985.XSHG", feature_date)
    else:
        raise ValueError("unsupported stock_pool: {}".format(stock_pool))

    if len(stock_list) == 0:
        return []

    st_data = get_extras("is_st", stock_list, count=1, end_date=feature_date)
    if st_data is not None and len(st_data) > 0:
        st_row = st_data.iloc[0]
        stock_list = [
            stock for stock in stock_list
            if (stock not in st_row.index) or pd.isnull(st_row[stock]) or (not bool(st_row[stock]))
        ]

    stock_list = filter_paused_stock_by_date(stock_list, feature_date)
    stock_list = delect_stop(stock_list, feature_date, n=MIN_LISTING_DAYS)
    return stock_list


def get_industry_bucket_map_for_data(stock_list, date):
    if len(stock_list) == 0:
        return {}
    try:
        industry_info = get_industry(stock_list, date=date)
    except Exception:
        return {stock: "UNKNOWN" for stock in stock_list}

    out = {}
    for stock in stock_list:
        info = industry_info.get(stock, {})
        bucket = None
        for key in ["sw_l1", "jq_l1", "zjw"]:
            sub = info.get(key, None)
            if isinstance(sub, dict):
                bucket = sub.get("industry_code") or sub.get("industry_name")
                if bucket:
                    break
        out[stock] = bucket if bucket else "UNKNOWN"
    return out


def get_factor_data(stock_list, date):
    if len(stock_list) == 0:
        return pd.DataFrame()

    df_factor = pd.DataFrame(index=stock_list)
    for fac_chunk in chunks(BASE_FACTOR_COLS, 20):
        try:
            factor_data = get_factor_values(
                securities=stock_list,
                factors=fac_chunk,
                count=1,
                end_date=date,
            )
        except Exception:
            factor_data = None

        for fac in fac_chunk:
            try:
                if factor_data is not None and fac in factor_data:
                    df_factor[fac] = factor_data[fac].iloc[0, :]
                else:
                    df_factor[fac] = np.nan
            except Exception:
                df_factor[fac] = np.nan
    return df_factor


def calc_ret(close_mat, days):
    if close_mat is None or close_mat.empty or len(close_mat) <= days:
        return pd.Series(index=close_mat.columns if close_mat is not None else [], dtype=float)
    return close_mat.iloc[-1] / close_mat.iloc[-days - 1] - 1


def calc_up_day_ratio(ret_mat, days):
    if ret_mat is None or ret_mat.empty:
        return pd.Series(dtype=float)
    return (ret_mat.tail(days) > 0).mean()


def calc_new_low_distance(close_mat, days):
    if close_mat is None or close_mat.empty:
        return pd.Series(dtype=float)
    last_close = close_mat.iloc[-1]
    min_close = close_mat.tail(days).min()
    return last_close / min_close - 1


def get_price_path_and_liquidity_data(stock_list, date, lookback=121, chunk_size=160):
    cols = V4_PRICE_PATH_COLS + TRADE_LIQUIDITY_COLS[:9]
    out_all = []
    for stock_chunk in chunks(stock_list, chunk_size):
        out = pd.DataFrame(index=stock_chunk, columns=cols, dtype=float)
        try:
            price_df = get_price(
                stock_chunk,
                end_date=date,
                frequency="daily",
                fields=["close", "high", "low", "volume", "money", "paused"],
                count=lookback,
                skip_paused=False,
                fq="pre",
                panel=False,
                fill_paused=True,
            )
        except Exception:
            price_df = None

        if price_df is None or price_df.empty:
            out_all.append(out)
            continue

        for col in ["close", "high", "low", "volume", "money", "paused"]:
            if col not in price_df.columns:
                price_df[col] = np.nan
        price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
        close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
        high_mat = price_df.pivot_table(index="time", columns="code", values="high").sort_index()
        low_mat = price_df.pivot_table(index="time", columns="code", values="low").sort_index()
        volume_mat = price_df.pivot_table(index="time", columns="code", values="volume").sort_index()
        money_mat = price_df.pivot_table(index="time", columns="code", values="money").sort_index()
        paused_mat = price_df.pivot_table(index="time", columns="code", values="paused").sort_index()

        ret_mat = close_mat.pct_change()
        last_close = close_mat.iloc[-1]
        ma20 = close_mat.tail(20).mean()
        ma60 = close_mat.tail(60).mean()
        money20 = money_mat.tail(20).mean()
        money60 = money_mat.tail(60).mean()
        volume20 = volume_mat.tail(20).mean()
        volume60 = volume_mat.tail(60).mean()

        out["px_ret_5"] = calc_ret(close_mat, 5)
        out["px_ret_20"] = calc_ret(close_mat, 20)
        out["px_ret_60"] = calc_ret(close_mat, 60)
        out["px_ret_120"] = calc_ret(close_mat, 120)
        out["px_close_to_ma20"] = last_close / ma20 - 1
        out["px_close_to_ma60"] = last_close / ma60 - 1
        out["px_ma20_to_ma60"] = ma20 / ma60 - 1
        out["px_volatility_20"] = ret_mat.tail(20).std()
        out["px_volatility_60"] = ret_mat.tail(60).std()
        out["px_drawdown_20"] = last_close / close_mat.tail(20).max() - 1
        out["px_drawdown_60"] = last_close / close_mat.tail(60).max() - 1
        out["px_drawdown_120"] = last_close / close_mat.tail(120).max() - 1
        out["px_up_day_ratio_20"] = calc_up_day_ratio(ret_mat, 20)
        out["px_new_high_distance_60"] = last_close / close_mat.tail(60).max() - 1
        out["px_new_low_distance_60"] = calc_new_low_distance(close_mat, 60)
        out["px_skew_20"] = ret_mat.tail(20).skew()
        out["px_kurt_20"] = ret_mat.tail(20).kurt()

        out["liq_money_mean_20"] = money20
        out["liq_money_mean_60"] = money60
        out["liq_money_ratio_20_60"] = money20 / money60 - 1
        out["liq_volume_mean_20"] = volume20
        out["liq_volume_ratio_20_60"] = volume20 / volume60 - 1
        out["liq_amplitude_mean_20"] = (high_mat.tail(20) / low_mat.tail(20) - 1).mean()
        out["liq_amplitude_mean_60"] = (high_mat.tail(60) / low_mat.tail(60) - 1).mean()
        out["liq_paused_count_20"] = paused_mat.tail(20).fillna(0).sum()
        out["liq_paused_count_60"] = paused_mat.tail(60).fillna(0).sum()

        out_all.append(out.replace([np.inf, -np.inf], np.nan))
        del price_df, close_mat, high_mat, low_mat, volume_mat, money_mat, paused_mat, ret_mat
        gc.collect()
    return pd.concat(out_all).reindex(index=stock_list)


def get_limit_state_data(stock_list, date, lookback=20, chunk_size=160):
    cols = [
        "liq_low_money_days_20",
        "liq_limit_up_count_20",
        "liq_limit_down_count_20",
        "liq_one_price_limit_count_20",
    ]
    out_all = []
    for stock_chunk in chunks(stock_list, chunk_size):
        out = pd.DataFrame(index=stock_chunk, columns=cols, dtype=float)
        try:
            price_df = get_price(
                stock_chunk,
                end_date=date,
                frequency="daily",
                fields=["close", "high", "low", "money", "paused", "high_limit", "low_limit"],
                count=lookback,
                skip_paused=False,
                fq=None,
                panel=False,
                fill_paused=True,
            )
        except Exception:
            price_df = None

        if price_df is None or price_df.empty:
            out_all.append(out)
            continue

        for col in ["close", "high", "low", "money", "paused", "high_limit", "low_limit"]:
            if col not in price_df.columns:
                price_df[col] = np.nan
        price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
        money_mat = price_df.pivot_table(index="time", columns="code", values="money").sort_index()
        close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
        high_mat = price_df.pivot_table(index="time", columns="code", values="high").sort_index()
        low_mat = price_df.pivot_table(index="time", columns="code", values="low").sort_index()
        high_limit_mat = price_df.pivot_table(index="time", columns="code", values="high_limit").sort_index()
        low_limit_mat = price_df.pivot_table(index="time", columns="code", values="low_limit").sort_index()

        money_q20 = money_mat.stack().quantile(0.20) if len(money_mat.stack().dropna()) else np.nan
        out["liq_low_money_days_20"] = (money_mat.tail(20) < money_q20).sum() if not pd.isnull(money_q20) else np.nan

        limit_up = close_mat >= (high_limit_mat * 0.999)
        limit_down = close_mat <= (low_limit_mat * 1.001)
        one_price = (high_mat <= low_mat * 1.0001) & (limit_up | limit_down)
        out["liq_limit_up_count_20"] = limit_up.tail(20).sum()
        out["liq_limit_down_count_20"] = limit_down.tail(20).sum()
        out["liq_one_price_limit_count_20"] = one_price.tail(20).sum()

        out_all.append(out.replace([np.inf, -np.inf], np.nan))
        del price_df, money_mat, close_mat, high_mat, low_mat, high_limit_mat, low_limit_mat
        gc.collect()
    return pd.concat(out_all).reindex(index=stock_list)


def get_market_context(date, benchmark=BENCHMARK, lookback=61):
    out = {
        "ctx_market_ret_20": np.nan,
        "ctx_market_ret_60": np.nan,
        "ctx_market_volatility_20": np.nan,
    }
    try:
        bench_df = get_price(
            benchmark,
            end_date=date,
            frequency="daily",
            fields=["close"],
            count=lookback,
            skip_paused=True,
            fq="pre",
        )
    except Exception:
        bench_df = None

    if bench_df is None or bench_df.empty or "close" not in bench_df.columns:
        return out
    close = bench_df["close"].dropna()
    if len(close) > 20:
        out["ctx_market_ret_20"] = close.iloc[-1] / close.iloc[-21] - 1
        out["ctx_market_volatility_20"] = close.pct_change().tail(20).std()
    if len(close) > 60:
        out["ctx_market_ret_60"] = close.iloc[-1] / close.iloc[-61] - 1
    return out


def attach_industry_context(factor_data, market_context):
    out = factor_data.copy()
    for col, value in market_context.items():
        out[col] = value

    for ret_col, ctx_col in [
        ("px_ret_20", "ctx_industry_ret_20"),
        ("px_ret_60", "ctx_industry_ret_60"),
    ]:
        out[ctx_col] = out.groupby("industry_bucket")[ret_col].transform("mean")

    out["ctx_stock_minus_industry_ret_20"] = out["px_ret_20"] - out["ctx_industry_ret_20"]
    out["ctx_stock_minus_industry_ret_60"] = out["px_ret_60"] - out["ctx_industry_ret_60"]
    out["ctx_stock_rank_industry_ret_20"] = out.groupby("industry_bucket")["px_ret_20"].rank(pct=True)
    out["ctx_stock_rank_industry_volatility_20"] = out.groupby("industry_bucket")["px_volatility_20"].rank(pct=True)
    return out


def get_forward_alpha(stock_list, date, next_date, benchmark):
    if len(stock_list) == 0:
        return pd.Series(dtype=float)

    price_df = get_price(
        stock_list,
        start_date=date,
        end_date=next_date,
        frequency="daily",
        fields=["close"],
        skip_paused=True,
        fq="pre",
        panel=False,
    )
    if price_df is None or price_df.empty:
        return pd.Series(dtype=float)

    price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
    close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
    if len(close_mat) < 2:
        return pd.Series(dtype=float)
    stock_ret = close_mat.iloc[-1] / close_mat.iloc[1] - 1

    bench_df = get_price(
        benchmark,
        start_date=date,
        end_date=next_date,
        frequency="daily",
        fields=["close"],
        skip_paused=True,
        fq="pre",
    )
    if bench_df is None or bench_df.empty or len(bench_df) < 2:
        return pd.Series(dtype=float)

    bench_ret = bench_df["close"].iloc[-1] / bench_df["close"].iloc[1] - 1
    return stock_ret - bench_ret


def add_core_factor_temporal_features(df):
    out = df.copy()
    out = out.sort_values(["rebalance_date", "stock"]).reset_index(drop=True)
    for factor in CORE_TEMPORAL_FACTORS:
        if factor not in out.columns:
            continue
        rank_col = "tmp_{}_rank".format(factor)
        out[rank_col] = out.groupby("rebalance_date")[factor].rank(pct=True)
        g_stock = out.groupby("stock")[rank_col]
        for lag in [1, 3]:
            col = "ts_{}_rank_chg_{}m".format(factor, lag)
            out[col] = out[rank_col] - g_stock.shift(lag)
        mean_col = "ts_{}_rank_mean_3m".format(factor)
        std_col = "ts_{}_rank_std_3m".format(factor)
        z_col = "ts_{}_rank_z_6m".format(factor)
        out[mean_col] = g_stock.transform(lambda s: s.shift(1).rolling(3, min_periods=2).mean())
        out[std_col] = g_stock.transform(lambda s: s.shift(1).rolling(3, min_periods=2).std())
        rolling_mean_6 = g_stock.transform(lambda s: s.shift(1).rolling(6, min_periods=3).mean())
        rolling_std_6 = g_stock.transform(lambda s: s.shift(1).rolling(6, min_periods=3).std())
        out[z_col] = (out[rank_col] - rolling_mean_6) / rolling_std_6
        out = out.drop(columns=[rank_col])
    return out


def build_v46_rebuild_dataset():
    require_joinquant_api()
    date_list = get_period_date("M", V4_DATA_START, V4_DATA_END_FOR_LABEL)
    print("V46 rebuild rebalance dates =", len(date_list), "|", V4_DATA_START, "->", V4_DATA_END_FOR_LABEL)

    all_rows = []
    for i, rebalance_date in enumerate(date_list[:-1]):
        next_date = date_list[i + 1]
        feature_date = get_previous_trade_date(rebalance_date)
        if feature_date is None:
            continue

        stock_list = get_stock(UNIVERSE_NAME, feature_date)
        if len(stock_list) == 0:
            continue

        jq_factor_data = get_factor_data(stock_list, feature_date)
        if jq_factor_data is None or jq_factor_data.empty:
            continue

        industry_map = get_industry_bucket_map_for_data(stock_list, feature_date)
        price_liq_data = get_price_path_and_liquidity_data(stock_list, feature_date)
        limit_data = get_limit_state_data(stock_list, feature_date)
        alpha = get_forward_alpha(stock_list, rebalance_date, next_date, BENCHMARK)
        if alpha.empty:
            continue

        factor_data = jq_factor_data.join(price_liq_data, how="left").join(limit_data, how="left")
        factor_data["stock"] = factor_data.index
        factor_data["industry_bucket"] = factor_data["stock"].map(industry_map).fillna("UNKNOWN")
        factor_data = attach_industry_context(factor_data, get_market_context(feature_date, BENCHMARK))
        factor_data["alpha_1m"] = alpha
        factor_data["rebalance_date"] = rebalance_date
        factor_data["feature_date"] = feature_date
        factor_data["next_date"] = next_date
        factor_data = factor_data.dropna(subset=["alpha_1m"]).copy()
        if len(factor_data) < 30:
            continue

        factor_data["alpha_rank_pct"] = factor_data["alpha_1m"].rank(pct=True, method="first")
        all_rows.append(factor_data.reset_index(drop=True))
        print(
            "  rebuilt {}/{} rebalance={} feature={} rows={}".format(
                i + 1, max(1, len(date_list) - 1), rebalance_date, feature_date, len(factor_data)
            )
        )

        del jq_factor_data, price_liq_data, limit_data, alpha, factor_data
        gc.collect()

    if len(all_rows) == 0:
        raise ValueError("V46 data rebuild produced no rows")
    df = pd.concat(all_rows, ignore_index=True)
    df = add_core_factor_temporal_features(df)
    df.to_csv(V4_DATA_FILE, index=False)
    print("V46 data rebuilt rows =", len(df), "saved ->", V4_DATA_FILE)
    return df





def maybe_rebuild_dataset():
    global DATA_PATH
    if not REBUILD_DATA:
        print("skip data rebuild; using DATA_PATH =", DATA_PATH)
        return None

    if os.path.exists(REBUILD_DATA_OUTPUT_PATH) and not REBUILD_FORCE_OVERWRITE:
        print("rebuilt data already exists; skip rebuild:", REBUILD_DATA_OUTPUT_PATH)
        if USE_REBUILT_DATA_FOR_TRAINING:
            DATA_PATH = REBUILD_DATA_OUTPUT_PATH
            print("DATA_PATH switched to existing rebuilt file:", DATA_PATH)
        return pd.read_csv(REBUILD_DATA_OUTPUT_PATH, nrows=5)

    print("rebuilding CSI800 monthly dataset")
    print("  start =", REBUILD_DATA_START, "end_for_label =", REBUILD_DATA_END_FOR_LABEL)
    rebuilt_df = build_v46_rebuild_dataset()

    for _col in ["rebalance_date", "feature_date", "next_date"]:
        if _col in rebuilt_df.columns:
            rebuilt_df[_col] = pd.to_datetime(rebuilt_df[_col])

    meta_cols = ["stock", "rebalance_date", "feature_date", "next_date", "alpha_1m", "alpha_rank_pct", "industry_bucket"]
    feature_cols_rebuilt = [c for c in rebuilt_df.columns if c not in meta_cols]
    date_min = rebuilt_df["rebalance_date"].min() if len(rebuilt_df) else pd.NaT
    date_max = rebuilt_df["rebalance_date"].max() if len(rebuilt_df) else pd.NaT

    rebuild_manifest = pd.DataFrame([{
        "data_file": REBUILD_DATA_OUTPUT_PATH,
        "rows": int(len(rebuilt_df)),
        "months": int(rebuilt_df["rebalance_date"].nunique()) if "rebalance_date" in rebuilt_df.columns else 0,
        "stock_count": int(rebuilt_df["stock"].nunique()) if "stock" in rebuilt_df.columns else 0,
        "feature_count": int(len(feature_cols_rebuilt)),
        "rebalance_date_min": str(date_min.date()) if not pd.isnull(date_min) else "",
        "rebalance_date_max": str(date_max.date()) if not pd.isnull(date_max) else "",
        "universe": UNIVERSE_NAME,
        "universe_index": UNIVERSE_INDEX,
        "benchmark": BENCHMARK,
        "min_listing_days": MIN_LISTING_DAYS,
        "rebuilt_by": "中证800_V51_factor_increment_mining实验.ipynb",
    }])
    rebuild_manifest.to_csv(REBUILD_MANIFEST_PATH, index=False)

    print("rebuilt rows =", len(rebuilt_df))
    print("rebalance date =", date_min, "->", date_max)
    print("feature count =", len(feature_cols_rebuilt))
    print("saved data ->", REBUILD_DATA_OUTPUT_PATH)
    print("saved manifest ->", REBUILD_MANIFEST_PATH)
    display(rebuild_manifest)

    if USE_REBUILT_DATA_FOR_TRAINING:
        DATA_PATH = REBUILD_DATA_OUTPUT_PATH
        print("DATA_PATH switched to rebuilt file:", DATA_PATH)

    return rebuilt_df


_rebuilt_preview_df = maybe_rebuild_dataset()
if _rebuilt_preview_df is not None:
    print("rebuild preview shape:", _rebuilt_preview_df.shape)


In [ ]:
ALL_FACTOR_GROUPS = {
    # Already validated: keep only as manual control, not in default active set.
    "validated_controls": [
        "turnover_volatility",
    ],

    # New cash-flow quality candidates. These test whether alpha_1m prefers cash conversion and balance-sheet cash safety
    # beyond V46's existing cash_flow_to_price_ratio / cash earnings factors.
    "cashflow_quality_candidates": [
        "cfo_to_ev",
        "cash_rate_of_sales",
        "goods_service_cash_to_operating_revenue_ttm",
        "net_operate_cash_flow_to_asset",
        "net_operate_cash_flow_to_net_debt",
        "net_operate_cash_flow_to_total_current_liability",
        "net_operate_cash_flow_to_operate_income",
    ],

    # New operating/profitability quality candidates. Avoid existing V46 columns where possible.
    "profit_quality_candidates": [
        "net_profit_ratio",
        "net_profit_to_total_operate_revenue_ttm",
        "operating_profit_ratio",
        "operating_profit_to_operating_revenue",
        "total_profit_to_cost_ratio",
        "profit_margin_ttm",
        "margin_stability",
        "maximum_margin",
    ],

    # New efficiency candidates. These may capture working-capital efficiency not visible in static value/quality factors.
    "operation_efficiency_candidates": [
        "inventory_turnover_rate",
        "account_receivable_turnover_rate",
        "accounts_payable_turnover_rate",
        "total_asset_turnover_rate",
        "current_asset_turnover_rate",
        "fixed_assets_turnover_rate",
        "asset_turnover_ttm",
        "equity_turnover_rate",
    ],

    # New growth-improvement candidates. These are deliberately tested separately because growth labels can be regime-sensitive.
    "growth_improvement_candidates": [
        "DEGM",
        "operating_profit_growth_rate",
        "operating_revenue_growth_rate",
        "net_profit_growth_rate",
        "total_profit_growth_rate",
        "net_operate_cashflow_growth_rate",
        "total_asset_growth_rate",
        "np_parent_company_owners_growth_rate",
    ],

    # Optional risk/shape follow-up excluding the already validated turnover_volatility.
    "risk_shape_candidates": [
        "residual_volatility",
        "Skewness60",
        "Skewness120",
        "VOL20",
        "DAVOL20",
        "VOL60",
        "VOL120",
    ],
}

# Default: dig new factors, not the already validated turnover_volatility.
# If OOM appears, run one group at a time, e.g. ["cashflow_quality_candidates"].
ACTIVE_GROUP_NAMES = [
    "cashflow_quality_candidates",
    "profit_quality_candidates",
    "operation_efficiency_candidates",
]

ACTIVE_FACTOR_COLS = []
for group_name in ACTIVE_GROUP_NAMES:
    if group_name not in ALL_FACTOR_GROUPS:
        raise ValueError("unknown factor group: " + str(group_name))
    for factor in ALL_FACTOR_GROUPS[group_name]:
        if factor not in ACTIVE_FACTOR_COLS:
            ACTIVE_FACTOR_COLS.append(factor)

# Do not silently re-test factors already in the V46 base set. The goal is incremental discovery.
BASE_FACTOR_SET = set(BASE_CANDIDATE_COLS)
ACTIVE_FACTOR_COLS = [f for f in ACTIVE_FACTOR_COLS if f not in BASE_FACTOR_SET]

ACTIVE_GROUP_TAG = "__".join(ACTIVE_GROUP_NAMES)
if len(ACTIVE_GROUP_TAG) > 80:
    ACTIVE_GROUP_TAG = "multi_new_factor_batch"
ACTIVE_OUT_DIR = os.path.join(OUT_DIR, ACTIVE_GROUP_TAG)
os.makedirs(ACTIVE_OUT_DIR, exist_ok=True)
ENRICHED_DATA_PATH = os.path.join(ACTIVE_OUT_DIR, "v51_enriched_dataset.csv")

print("active groups:", ACTIVE_GROUP_NAMES)
print("active factors:", len(ACTIVE_FACTOR_COLS), ACTIVE_FACTOR_COLS)
print("ACTIVE_OUT_DIR =", ACTIVE_OUT_DIR)
print("ENRICHED_DATA_PATH =", ENRICHED_DATA_PATH)


In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def chunks(seq, size):
    seq = list(seq)
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def safe_stats(values):
    s = pd.Series(values).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return {"mean": np.nan, "std": np.nan, "ir": np.nan, "hit_rate": np.nan, "months": 0}
    std = s.std()
    return {
        "mean": float(s.mean()),
        "std": float(std) if not pd.isnull(std) else np.nan,
        "ir": float(s.mean() / std) if (not pd.isnull(std) and std > 0) else np.nan,
        "hit_rate": float((s > 0).mean()),
        "months": int(len(s)),
    }


def max_drawdown_from_returns(returns):
    s = pd.Series(returns).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return np.nan
    nav = (1.0 + s).cumprod()
    return float((nav / nav.cummax() - 1.0).min())


def cross_section_rank_pct(s):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    out = pd.Series(index=s.index, dtype=float)
    valid = s.dropna()
    if len(valid) == 0:
        return out
    out.loc[valid.index] = valid.rank(method="average") / float(len(valid))
    return out


def eval_topn(month_df, score_col, n):
    d = month_df[["stock", "rebalance_date", TARGET_COL, score_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) == 0:
        return None
    top = d.sort_values(score_col, ascending=False).head(min(n, len(d)))
    return {"mean_alpha": float(top[TARGET_COL].mean()), "count": int(len(top)), "targets": ",".join(list(top["stock"]))}


In [ ]:
def load_base_df(path):
    if not os.path.exists(path):
        raise IOError("DATA_PATH not found: " + path)
    df = pd.read_csv(path)
    if "code" in df.columns and "stock" not in df.columns:
        df = df.rename(columns={"code": "stock"})
    for col in ["rebalance_date", "feature_date", "next_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col]).dt.normalize()
    if TARGET_COL not in df.columns:
        raise ValueError("missing target: " + TARGET_COL)
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=["stock", "rebalance_date", "feature_date", TARGET_COL]).copy()
    return df


def fetch_factor_values_one_date(stock_list, factor_list, feature_date):
    try:
        data_api = get_factor_values
    except NameError:
        raise RuntimeError("fetching candidate factors requires JoinQuant jqfactor.get_factor_values")

    date_str = pd.Timestamp(feature_date).strftime("%Y-%m-%d")
    rows = []
    for stock_chunk in chunks(stock_list, STOCK_CHUNK_SIZE):
        tmp = pd.DataFrame(index=stock_chunk)
        for factor_chunk in chunks(factor_list, FACTOR_CHUNK_SIZE):
            try:
                data = data_api(
                    securities=stock_chunk,
                    factors=factor_chunk,
                    end_date=date_str,
                    count=1,
                )
            except Exception as err:
                print("factor fetch failed:", date_str, len(stock_chunk), factor_chunk, err)
                data = None
            for factor in factor_chunk:
                try:
                    if data is not None and factor in data:
                        tmp[factor] = data[factor].iloc[-1, :].reindex(stock_chunk)
                    else:
                        tmp[factor] = np.nan
                except Exception:
                    tmp[factor] = np.nan
            del data
        tmp["stock"] = tmp.index
        tmp["feature_date"] = pd.Timestamp(feature_date)
        rows.append(tmp.reset_index(drop=True))
        del tmp
        gc.collect()
    if not rows:
        out = pd.DataFrame({"stock": stock_list})
        out["feature_date"] = pd.Timestamp(feature_date)
        for factor in factor_list:
            out[factor] = np.nan
        return out
    out = pd.concat(rows, ignore_index=True, sort=False)
    del rows
    gc.collect()
    return out


def build_enriched_dataset(base_df, factor_cols):
    key_cols = ["stock", "feature_date"]
    existing_factor_cols = [f for f in factor_cols if f in base_df.columns]
    missing_factor_cols = [f for f in factor_cols if f not in base_df.columns]
    keep_cols = unique_keep_order(["stock", "rebalance_date", "feature_date", "next_date", TARGET_COL] + BASE_CANDIDATE_COLS + existing_factor_cols)
    compact = base_df[[c for c in keep_cols if c in base_df.columns]].copy()

    if len(missing_factor_cols) == 0:
        print("all active factors already exist in DATA_PATH")
        return compact

    pairs = compact[key_cols].drop_duplicates().copy()
    out_parts = []
    feature_dates = sorted(pd.to_datetime(pairs["feature_date"].dropna().unique()))
    for i, dt in enumerate(feature_dates):
        stock_list = list(pairs[pairs["feature_date"] == dt]["stock"].drop_duplicates())
        print("fetch", i + 1, "/", len(feature_dates), pd.Timestamp(dt).strftime("%Y-%m-%d"), "stocks", len(stock_list), "factors", missing_factor_cols)
        one = fetch_factor_values_one_date(stock_list, missing_factor_cols, dt)
        out_parts.append(one)
        del one, stock_list
        if (i + 1) % GC_EVERY_N_FETCH_DATES == 0:
            gc.collect()

    factor_df = pd.concat(out_parts, ignore_index=True, sort=False) if out_parts else pd.DataFrame(columns=key_cols + missing_factor_cols)
    del out_parts, pairs
    gc.collect()
    factor_df["feature_date"] = pd.to_datetime(factor_df["feature_date"]).dt.normalize()
    enriched = compact.merge(factor_df, on=key_cols, how="left", suffixes=("", "_newdup"), sort=False)
    dup_cols = [c for c in enriched.columns if c.endswith("_newdup")]
    if dup_cols:
        enriched = enriched.drop(columns=dup_cols)
    dup_count = int(enriched.duplicated(["stock", "feature_date"]).sum())
    if dup_count != 0:
        raise ValueError("enriched dataset duplicated stock-feature_date rows: " + str(dup_count))
    return enriched


base_df = load_base_df(DATA_PATH)
print("base:", base_df.shape, base_df["rebalance_date"].min(), base_df["rebalance_date"].max())

if os.path.exists(ENRICHED_DATA_PATH) and not FORCE_REFETCH_FACTORS:
    enriched_df = load_base_df(ENRICHED_DATA_PATH)
    missing_cols = [f for f in ACTIVE_FACTOR_COLS if f not in enriched_df.columns]
    dup_count = int(enriched_df.duplicated(["stock", "feature_date"]).sum())
    if missing_cols or dup_count:
        print("invalid cache; rebuild required:", "missing_cols=", missing_cols, "dup_count=", dup_count)
        enriched_df = build_enriched_dataset(base_df, ACTIVE_FACTOR_COLS)
        enriched_df.to_csv(ENRICHED_DATA_PATH, index=False)
    else:
        print("loaded enriched cache:", ENRICHED_DATA_PATH, enriched_df.shape)
elif FETCH_FACTORS_IF_CACHE_MISSING:
    enriched_df = build_enriched_dataset(base_df, ACTIVE_FACTOR_COLS)
    enriched_df.to_csv(ENRICHED_DATA_PATH, index=False)
    print("saved enriched cache:", ENRICHED_DATA_PATH, enriched_df.shape)
else:
    raise IOError("enriched cache missing and FETCH_FACTORS_IF_CACHE_MISSING=False: " + ENRICHED_DATA_PATH)

del base_df
gc.collect()

coverage_rows = []
for factor in ACTIVE_FACTOR_COLS:
    coverage_rows.append({"factor": factor, "coverage": float(enriched_df[factor].notnull().mean()) if factor in enriched_df.columns else 0.0})
coverage_df = pd.DataFrame(coverage_rows).sort_values("coverage", ascending=False)
coverage_df.to_csv(os.path.join(ACTIVE_OUT_DIR, "v51_factor_coverage.csv"), index=False)
print(coverage_df.to_string(index=False))


In [ ]:
work_df = enriched_df[(enriched_df["rebalance_date"] >= pd.Timestamp(TRAIN_START)) & (enriched_df["rebalance_date"] <= pd.Timestamp(TRAIN_END))].copy()
del enriched_df
gc.collect()

available_factor_cols = [f for f in ACTIVE_FACTOR_COLS if f in work_df.columns and work_df[f].notnull().mean() > 0.05]
missing_or_sparse = [f for f in ACTIVE_FACTOR_COLS if f not in available_factor_cols]
months = sorted(pd.to_datetime(work_df["rebalance_date"].dropna().unique()))
eval_months = months[MIN_HISTORY_MONTHS:]

print("work:", work_df.shape, work_df["rebalance_date"].min(), work_df["rebalance_date"].max(), "months", len(months))
print("eval months:", len(eval_months), eval_months[0] if eval_months else None, eval_months[-1] if eval_months else None)
print("available factors:", available_factor_cols)
print("missing or sparse:", missing_or_sparse)


In [ ]:
def summarize_prior_ic(hist_df, factor):
    rows = []
    for dt, g in hist_df.groupby("rebalance_date"):
        rows.append(safe_rank_ic(g[factor], g[TARGET_COL]))
    st = safe_stats(rows)
    return st


single_month_rows = []
for factor_i, factor in enumerate(available_factor_cols):
    for month_i, month in enumerate(eval_months):
        hist_df = work_df[work_df["rebalance_date"] < month]
        month_df = work_df[work_df["rebalance_date"] == month].copy()
        if hist_df["rebalance_date"].nunique() < MIN_HISTORY_MONTHS or month_df.empty:
            del month_df
            continue
        st = summarize_prior_ic(hist_df, factor)
        ic_mean = st["mean"]
        direction = 1.0 if (pd.isnull(ic_mean) or ic_mean >= 0) else -1.0
        score_col = "score_" + factor
        month_df[score_col] = cross_section_rank_pct(month_df[factor]) * direction
        for n in TOP_LIST:
            r = eval_topn(month_df, score_col, n)
            if r is None:
                continue
            single_month_rows.append({
                "rebalance_date": month,
                "factor": factor,
                "topn": int(n),
                "mean_alpha": r["mean_alpha"],
                "count": r["count"],
                "direction": direction,
                "prior_ic_mean": ic_mean,
                "prior_ic_ir": st["ir"],
                "targets": r["targets"],
            })
        del month_df
    print("single-factor audit processed:", factor_i + 1, "/", len(available_factor_cols), factor)
    gc.collect()

single_month_df = pd.DataFrame(single_month_rows)
summary_rows = []
if not single_month_df.empty:
    for (factor, topn), g in single_month_df.groupby(["factor", "topn"]):
        st = safe_stats(g["mean_alpha"])
        summary_rows.append({
            "factor": factor,
            "topn": int(topn),
            "months": st["months"],
            "mean_monthly_alpha": st["mean"],
            "monthly_alpha_ir": st["ir"],
            "win_rate": st["hit_rate"],
            "max_drawdown": max_drawdown_from_returns(g.sort_values("rebalance_date")["mean_alpha"]),
        })

single_summary_df = pd.DataFrame(summary_rows)
if not single_summary_df.empty:
    single_summary_df = single_summary_df.sort_values(["topn", "mean_monthly_alpha"], ascending=[True, False])

single_month_df.to_csv(os.path.join(ACTIVE_OUT_DIR, "v51_single_factor_monthly.csv"), index=False)
single_summary_df.to_csv(os.path.join(ACTIVE_OUT_DIR, "v51_single_factor_summary.csv"), index=False)
print(single_summary_df[single_summary_df["topn"] == 10].head(50).to_string(index=False))


In [ ]:
year_rows = []
if not single_month_df.empty:
    tmp = single_month_df.copy()
    tmp["year"] = pd.to_datetime(tmp["rebalance_date"]).dt.year
    for (factor, topn, year), g in tmp.groupby(["factor", "topn", "year"]):
        st = safe_stats(g["mean_alpha"])
        year_rows.append({
            "factor": factor,
            "topn": int(topn),
            "year": int(year),
            "months": st["months"],
            "mean_alpha": st["mean"],
            "ir": st["ir"],
            "win_rate": st["hit_rate"],
        })

yearly_df = pd.DataFrame(year_rows)
if not yearly_df.empty:
    yearly_df = yearly_df.sort_values(["factor", "topn", "year"])
yearly_df.to_csv(os.path.join(ACTIVE_OUT_DIR, "v51_single_factor_yearly.csv"), index=False)
print(yearly_df[yearly_df["topn"] == 10].to_string(index=False))


In [ ]:
def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    feature_cols = [c for c in feature_cols if c in train_df.columns]
    if len(feature_cols) == 0:
        return []
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []
    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)
    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0])
            remove.extend(comp[1:])
    return keep, remove


def split_inner_train_valid(train_df):
    months = sorted(pd.to_datetime(train_df["rebalance_date"].dropna().unique()))
    n_valid = max(INNER_VALID_MIN_MONTHS, int(round(len(months) * INNER_VALID_FRAC)))
    valid_months = set(months[-min(n_valid, max(1, len(months) - 1)):])
    fit = train_df[~train_df["rebalance_date"].isin(valid_months)].copy()
    valid = train_df[train_df["rebalance_date"].isin(valid_months)].copy()
    if fit.empty:
        fit = train_df.copy()
        valid = train_df.copy()
    return fit, valid


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d[feature_cols].replace([np.inf, -np.inf], np.nan)
    y = d[target_col].astype(float)
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values


def train_fixed_lgb(train_df, feature_cols, target_col):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    X_train, y_train, fill_values = prepare_xy(train_df, feature_cols, target_col)
    model = lgb.train(
        params,
        lgb.Dataset(X_train, label=y_train),
        num_boost_round=max(1, int(FIXED_NUM_BOOST_ROUND)),
        valid_sets=[lgb.Dataset(X_train, label=y_train)],
        valid_names=["train"],
        verbose_eval=False,
    )
    return model, fill_values, int(len(X_train))


def calc_inner_rank_ic(model, inner_valid_df, feature_cols, target_col, fill_values):
    X_valid, y_valid, _ = prepare_xy(inner_valid_df, feature_cols, target_col, fill_values)
    pred = np.asarray(model.predict(X_valid[feature_cols], num_iteration=FIXED_NUM_BOOST_ROUND)).reshape(-1)
    return safe_rank_ic(y_valid, pred)


def export_direct_bundle(model_name, research_version, train_df, candidate_cols, new_factor):
    fit_df, inner_valid_df = split_inner_train_valid(train_df)
    feature_cols, removed_cols = select_features_train_only(fit_df, candidate_cols)
    new_factor_selected = bool(new_factor == "" or new_factor in feature_cols)
    if (new_factor != "") and (not new_factor_selected) and (not EXPORT_UNSELECTED_MODELS):
        return {
            "model_file": model_name,
            "research_version": research_version,
            "new_factor": new_factor,
            "exported": False,
            "skip_reason": "new_factor_removed_by_corr_filter",
            "feature_count": len(feature_cols),
            "new_factor_selected": False,
        }

    model, fill_values, train_rows = train_fixed_lgb(train_df, feature_cols, TARGET_COL)
    inner_rank_ic = calc_inner_rank_ic(model, inner_valid_df, feature_cols, TARGET_COL, fill_values)
    bundle = {
        "objective": "v210_refit_fixed_iter_overlay",
        "research_version": research_version,
        "benchmark": BENCHMARK,
        "train_start": TRAIN_START,
        "train_end": TRAIN_END,
        "label_end": LABEL_END,
        "require_label_end_within_train": bool(REQUIRE_LABEL_END_WITHIN_TRAIN),
        "target_col": TARGET_COL,
        "target_note": "V51 factor increment mining: direct alpha_1m regression",
        "data_file": DATA_PATH,
        "protocol": "v51_factor_increment_fixed_iter_direct",
        "training_policy": "expanding",
        "param_set": "v46_ff10_plus_single_factor",
        "final_role": "v51_factor_increment",
        "base_params": BASE_PARAMS_FF10,
        "base_model": model,
        "base_feature_cols": list(feature_cols),
        "base_fill_values": dict(fill_values),
        "base_best_iter": int(FIXED_NUM_BOOST_ROUND),
        "es_best_iter": np.nan,
        "model_iter": int(FIXED_NUM_BOOST_ROUND),
        "fixed_iter": int(FIXED_NUM_BOOST_ROUND),
        "base_inner_metrics": {"inner_rank_ic": float(inner_rank_ic) if not pd.isnull(inner_rank_ic) else np.nan},
        "base_removed_features": list(removed_cols),
        "increment_factor": new_factor,
        "increment_factor_selected": bool(new_factor_selected),
        "residual_model": None,
        "residual_feature_cols": [],
        "residual_fill_values": {},
        "overlay_weight": 0.0,
        "overlay_mode": "direct",
        "top_n_candidates": TOP_N_CANDIDATES,
        "stock_num": STOCK_NUM,
        "industry_cap_ratio": INDUSTRY_CAP_RATIO,
        "requires_v4_feature_adapter": True,
        "uses_time_weight": False,
        "uses_current_valid_for_training": False,
    }
    out_path = os.path.join(ACTIVE_OUT_DIR, model_name)
    with open(out_path, "wb") as f:
        pickle.dump(bundle, f, protocol=2)
    del model
    gc.collect()
    return {
        "model_file": model_name,
        "model_path": out_path,
        "research_version": research_version,
        "new_factor": new_factor,
        "exported": True,
        "skip_reason": "",
        "train_rows": int(train_rows),
        "train_months": int(train_df["rebalance_date"].nunique()),
        "feature_count": len(feature_cols),
        "new_factor_selected": bool(new_factor_selected),
        "removed_feature_count": len(removed_cols),
        "fixed_iter": int(FIXED_NUM_BOOST_ROUND),
        "inner_rank_ic": inner_rank_ic,
    }


In [ ]:
train_df = work_df.copy()
if REQUIRE_LABEL_END_WITHIN_TRAIN:
    train_df = train_df[pd.to_datetime(train_df["next_date"]) <= pd.Timestamp(LABEL_END)].copy()

export_rows = []

if EXPORT_BASELINE_MODEL:
    print("exporting V46 baseline in same env")
    row = export_direct_bundle(
        "model_candidate_v51_v46_baseline_alpha_direct_fixed120.pkl",
        "candidate_v51_v46_baseline_alpha_direct_fixed120",
        train_df,
        BASE_CANDIDATE_COLS,
        "",
    )
    export_rows.append(row)

if EXPORT_INCREMENT_MODELS:
    for factor in available_factor_cols:
        print("exporting V46 +", factor)
        model_file = "model_candidate_v51_v46_plus_{}_alpha_direct_fixed120.pkl".format(factor)
        research_version = "candidate_v51_v46_plus_{}_alpha_direct_fixed120".format(factor)
        row = export_direct_bundle(
            model_file,
            research_version,
            train_df,
            BASE_CANDIDATE_COLS + [factor],
            factor,
        )
        export_rows.append(row)
        gc.collect()

export_manifest_df = pd.DataFrame(export_rows)
export_manifest_path = os.path.join(ACTIVE_OUT_DIR, "v51_model_export_manifest.csv")
export_manifest_df.to_csv(export_manifest_path, index=False)
print("saved export manifest:", export_manifest_path)
display(export_manifest_df)


In [ ]:
summary_xlsx = os.path.join(ACTIVE_OUT_DIR, "v51_factor_increment_summary.xlsx")
try:
    with pd.ExcelWriter(summary_xlsx) as writer:
        coverage_df.to_excel(writer, "coverage", index=False)
        single_summary_df.to_excel(writer, "single_summary", index=False)
        single_month_df.to_excel(writer, "single_monthly", index=False)
        yearly_df.to_excel(writer, "yearly", index=False)
        export_manifest_df.to_excel(writer, "model_exports", index=False)
    print("saved xlsx:", summary_xlsx)
except Exception as err:
    print("xlsx export skipped:", err)

print("saved files:")
for fn in sorted(os.listdir(ACTIVE_OUT_DIR)):
    print("  ", fn)


## 结论填写区

回测后重点回填：

- 单因子 OOS 是否为正，是否稳定到 top10/top30。
- `V46 + factor` 是否优于同环境导出的 V46 baseline。
- 如果改善，改善来自收益提升、回撤下降、胜率提升，还是板块暴露变化。
- 只保留能通过线上 pkl 回测验证的因子。